# Forecasting monthly groundwater levels in Bangladesh
**Mojtaba · 13:30–14:45 · Hands-on session 4**

Does filling gaps improve a six-month forecast at Haluaghat? Does satellite water storage add useful information? Compare both against the same seasonal baseline.

Open this notebook in Colab and choose **Runtime → Run all**. The four workshop data files are bundled inside this notebook from workshop revision 2fcd6dd. No Drive mount, repository clone, or earlier session is needed. Internet access is required to install packages and download the pretrained weights. The default runs three forecast windows on CPU; GPU fine-tuning is optional.

In [ ]:
from time import perf_counter
setup_start = perf_counter()
model_name = "Chronos-2 Small"  # Choose "TiRex-2" and rerun all cells to switch families.
%pip -q install "darts[torch]" huggingface_hub h5py scikit-learn
if model_name == "TiRex-2":
    %pip -q install tirex-2
install_seconds = perf_counter() - setup_start

**While Colab installs the package (2 minutes)**

Open [TSFM.ai](https://tsfm.ai/) and look up Chronos-2 in the model catalog. Ask the group: *If we only know the neighboring well’s readings up to today, what capability should we look for?* Look for support for **past-only covariates**. This is why we can use Chronos-2 Small for both forecasts in today’s experiment.

For a quick demo, the site’s playground runs Chronos-Bolt Tiny and Mini in the browser. Return to Colab for the groundwater exercise.

## 13:30 · Meet the well
The original monthly BWDB table keeps missing readings visible. The signed levels follow the workshop convention: a more negative value means a deeper water table. We keep a regular monthly calendar even when a reading is missing.

In [ ]:
#@title Load bundled workshop data (run automatically)
import base64
import io
import zipfile

archive = zipfile.ZipFile(io.BytesIO(base64.b64decode(
'...'
)))

In [ ]:
import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

def monthly_table(name):
    with archive.open(name) as src:
        return pd.read_csv(src, index_col="Date", parse_dates=True).sort_index().asfreq("MS")

levels = monthly_table("BWDB_gwl_ts_monthly.csv")
published_fill = monthly_table("BWDB_gwl_ts_monthly_filled.csv")
with archive.open("BWDB_gwl_station_info.csv") as src:
    stations = pd.read_csv(src).set_index("WellID")
display(stations.loc[["MY050", "JA001", "MY055"], ["District", "Upazila", "Latitude", "Longitude"]])

In [ ]:
target = "MY050"
neighbor = "JA001"  # Try MY055, another well near Haluaghat.
horizon = 6
origins = pd.date_range("2010-01-01", periods=7, freq="6MS")  # first hidden month
full_backtest = False  # Enable to evaluate all seven windows.
if not full_backtest:
    origins = origins[[0, 3, 6]]  # Include both January and July starts.
example = origins[-1]

def history(origin):
    return levels.loc[levels.index < origin, [target, neighbor]].tail(120)

def seasonal_forecast(origin):
    dates = pd.date_range(origin, periods=horizon, freq="MS")
    prior_year = dates - pd.DateOffset(years=1)
    return pd.Series(levels[target].reindex(prior_year).to_numpy(), index=dates)

past = history(example)
ax = past[target].tail(36).plot(x_compat=True, marker=".", label="Observed")
seasonal_forecast(example).plot(x_compat=True, ax=ax, marker="o", label="Same month last year")
ax.axvline(example, color="gray", linestyle="--")
ax.set(ylabel="Groundwater level (m relative to ground)", title=f"{target} · predict the next six months")
ax.legend()
plt.show()

Before revealing the readings, sketch your forecast. In which month would you expect the water table to be deepest?

In [ ]:
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.ensemble import RandomForestRegressor

with h5py.File(io.BytesIO(archive.read("TWSA_M_2002_2017_BGD_025.nc4")), "r") as grid:
    site = stations.loc[target]
    iy = np.abs(grid["lat"][:] - site.Latitude).argmin()
    ix = np.abs(grid["lon"][:] - site.Longitude).argmin()
    epoch = grid["time"].attrs["units"].decode().split(" since ")[1]
    months = (pd.Timestamp(epoch) + pd.to_timedelta(grid["time"][:], unit="D")).to_period("M").to_timestamp()
    grace = pd.Series(grid["twsa"][:, iy, ix], index=months, name="GRACE TWSA").asfreq("MS")
    print(f"Nearest GRACE cell: {grid['lat'][iy]:.2f}°, {grid['lon'][ix]:.2f}°")

# Demonstration only: the published values help visualize what the earlier session filled.
published = published_fill[target].reindex(past.index)
ax = past[target].plot(x_compat=True, label="Observed")
added = published[past[target].isna()].dropna()
ax.scatter(added.index, added, color="orange", label="Published gap fills")
ax.set_ylabel("Groundwater level (m relative to ground)")
ax.legend()
plt.show()

def fill_history(origin):
    frame = levels.loc[levels.index < origin].tail(120).dropna(axis=1, how="all")
    imputer = IterativeImputer(estimator=RandomForestRegressor(
        n_estimators=30, max_features="sqrt", min_samples_leaf=5, n_jobs=1,
        random_state=42), max_iter=3, random_state=42)
    values = pd.DataFrame(imputer.fit_transform(frame), index=frame.index, columns=frame.columns)
    first, last = frame[target].first_valid_index(), frame[target].last_valid_index()
    return values[target].where((frame.index >= first) & (frame.index <= last))

filled_histories = {origin: fill_history(origin) for origin in origins}

## 13:40 · Forecast from one well
Chronos-2 learned patterns during pretraining. Here its weights stay fixed. The input is up to ten years of monthly history, and the output is six monthly forecasts. The median is our point forecast, and the 10th and 90th percentiles describe the model's central 80% prediction interval. Darts uses fit() to initialize each zero-shot input configuration. Model weights stay frozen.

In [ ]:
import torch
from darts import TimeSeries
from darts.models import Chronos2Model
from darts.utils.likelihood_models import QuantileRegression
from darts.utils.timeseries_generation import datetime_attribute_timeseries

if model_name == "Chronos-2 Small":
    model_class, model_id = Chronos2Model, "autogluon/chronos-2-small"
elif model_name == "TiRex-2":
    from darts.models import TiRex2Model
    model_class, model_id = TiRex2Model, "NX-AI/TiRex-2"
else:
    raise ValueError("Choose Chronos-2 Small or TiRex-2 in the setup cell.")

device = "cuda" if torch.cuda.is_available() else "cpu"
quantiles = [0.1, 0.5, 0.9]

def make_model(**options):
    return model_class(input_chunk_length=(1, 120), output_chunk_length=horizon,
                       hub_model_name=model_id, likelihood=QuantileRegression(quantiles),
                       pl_trainer_kwargs={"accelerator": "gpu" if device == "cuda" else "cpu",
                                          "devices": 1}, **options)

def inputs(origin, mode):
    past = history(origin)
    series = TimeSeries.from_series(past[target].astype("float32"))
    if mode in ("filled", "grace"):
        series = TimeSeries.from_series(filled_histories[origin].astype("float32"))
        if mode == "filled":
            return series, {}
        # Assumed two-month publication lag; historical release dates are unavailable.
        observed = grace.shift(2).reindex(past.index).astype("float32")
        assert observed.notna().any(), "No GRACE observations in this history"
        return series, {"past_covariates": TimeSeries.from_series(observed)}
    if mode == "neighbor":
        return series, {"past_covariates": TimeSeries.from_series(past[neighbor].astype("float32"))}
    if mode == "calendar":
        calendar = datetime_attribute_timeseries(series, "month", cyclic=True,
                                                 add_length=horizon, dtype="float32")
        return series, {"future_covariates": calendar}
    if mode == "target":
        return series, {}
    raise ValueError(f"Unknown input mode: {mode}")

models = {}
forecast_times = []
def forecast(origin, mode="target", tuned=None):
    started = perf_counter()
    series, covariates = inputs(origin, mode)
    if tuned is None:
        if mode not in models:
            # Initialize each input schema once using only pre-evaluation history.
            initial, initial_cov = inputs(origins[0], mode)
            model = make_model()
            model.fit(initial, **initial_cov)
            models[mode] = model
        model = models[mode]
    else:
        model = tuned
    pred = model.predict(horizon, series=series, **covariates, predict_likelihood_parameters=True)
    dates = pd.date_range(origin, periods=horizon, freq="MS")
    assert pred.time_index.equals(dates)
    forecast_times.append(perf_counter() - started)
    return pd.DataFrame(pred.values(), index=dates, columns=[str(q) for q in quantiles])

single = forecast(example)
display(single.round(2))

## 13:55 · Compare gap filling and satellite information
Compare original well readings, cutoff-specific gap-filled readings, and the same gap-filled readings with GRACE. The satellite input is a regional terrestrial water-storage anomaly at the nearest grid cell. We assume two months of reporting delay for this retrospective exercise. Each imputer uses only the 120-month input history; the smaller forest keeps the classroom run economical.

In [ ]:
filled_pred = forecast(example, "filled")
grace_pred = forecast(example, "grace")
ax = past[target].tail(24).plot(x_compat=True, color="black", label="Target history")
for pred, label in [(single, "Target only"), (filled_pred, "Gap-filled"), (grace_pred, "Gap-filled + GRACE")]:
    line, = ax.plot(pred.index, pred["0.5"], marker="o", label=label)
    ax.fill_between(pred.index, pred["0.1"], pred["0.9"], color=line.get_color(), alpha=0.12)
ax.set(ylabel="Groundwater level (m relative to ground)", title=f"Forecast at {target} using history through {past.index[-1]:%B %Y}")
ax.legend()
plt.show()

## 14:15 · Reveal the observations and test accuracy
Compare mean absolute error (MAE) in metres. An error of 0.5 m counts equally whether the forecast was too shallow or too deep. Score all methods on the same observed months with an available seasonal baseline. Then repeat across the selected non-overlapping six-month windows.

In [ ]:
def comparison(origin, single, filled_pred, grace_pred):
    table = pd.DataFrame({"Seasonal": seasonal_forecast(origin), "Target only": single["0.5"],
                          "Gap-filled": filled_pred["0.5"], "Gap-filled + GRACE": grace_pred["0.5"]})
    table["Observed"] = levels[target].reindex(table.index)
    return table

shown = comparison(example, single, filled_pred, grace_pred)
ax = shown.plot(marker="o", ylabel="Groundwater level (m relative to ground)",
                title=f"{target} · forecasts and observations")
plt.show()
display(shown.round(2))

In [ ]:
windows = {}
for origin in origins:
    if origin == example:
        windows[origin] = shown
    else:
        windows[origin] = comparison(origin, forecast(origin),
                                     forecast(origin, "filled"), forecast(origin, "grace"))

results = pd.concat(windows, names=["Forecast start", "Month"]).dropna()
errors = results.drop(columns="Observed").sub(results["Observed"], axis=0).abs()
by_window = errors.groupby(level="Forecast start").mean()
by_window["Months scored"] = errors.groupby(level="Forecast start").size()
display(by_window.round(2))
display(errors.mean().rename("MAE across scored months (m)").round(2))

print(f"{model_id} on {device} | package setup: {install_seconds:.1f} s")
print(f"{len(forecast_times)} forecasts including first model loads: {sum(forecast_times):.1f} s")

## 14:35 · Discuss the result
Did filling gaps lower MAE? Did adding GRACE improve the same filled forecast? Check each window before interpreting the average.

Optional: call `forecast(example, "neighbor")` or `forecast(example, "calendar")` to try neighboring-well history or a known-future calendar on the original target history.

**Leaderboard discussion (3 minutes).** Which result would help you choose a model for another well? Open the [TSFM.ai benchmark archive](https://tsfm.ai/benchmarks). These examples were checked on November 21, 2026.

| Benchmark | Task | Leading entry on the displayed metric |
|---|---|---|
| FEV Bench | Point forecast across real-world tasks | TimeRSI-1.5, 41.44 MASE skill score (higher is better) |
| GIFT-Eval | Point forecast rank across diverse settings | Albert, 16.45 average MASE rank (lower is better) |
| QuitoBench | Changing trend and seasonality in application traffic | CrossFormer, 2.86 mean rank, followed by full Chronos-2 at 3.36 |
| BOOM | Probabilistic forecasts of infrastructure telemetry | Toto 2.0 variants occupy the first three places |
| Impermanent | Forecasts scored as new observations arrive | ZeroModel, 2.21 mean MASE/CRPS rank |
| ARFBench | Answering questions about anomalies in charts | Toto-1.0-QA-Experimental 32B, 63.9% accuracy among individual models |
| TIME | Performance by task and temporal pattern | Select a task and metric on the official leaderboard |

Start with FEV Bench. Its seasonal baseline connects to our same-month-last-year forecast. Then look at QuitoBench: would second place for full Chronos-2 on application traffic predict how Chronos-2 Small performs at MY050? Use the local error table to explain what evidence you would want before choosing a model for another well.

### Optional · Fine-tune on earlier groundwater records
On a GPU, enable the switch below. Use readings strictly before the earliest evaluation window to update a copy of the model. Compare it with the frozen target-only model on exactly the same scored months. This short training run demonstrates the workflow; the class can test whether it improves accuracy.

In [ ]:
run_fine_tuning = False
if run_fine_tuning:
    if device != "cuda":
        raise RuntimeError("Choose a GPU runtime for the optional fine-tuning exercise.")
    training = levels.loc[levels.index < origins[0]]
    if model_name != "Chronos-2 Small":
        raise ValueError("Select Chronos-2 Small for this fine-tuning exercise.")
    # Use only observed, contiguous training stretches with sufficient history.
    stretches = []
    for _, well in training.items():
        for _, segment in well.groupby(well.isna().cumsum()):
            segment = segment.dropna()
            if len(segment) >= 120 + horizon:
                stretches.append(TimeSeries.from_series(segment.astype("float32")))
    tuned = make_model(enable_finetuning=True, n_epochs=1, batch_size=4)
    tuned.fit(stretches)
    tuned_values = pd.concat({o: forecast(o, tuned=tuned)["0.5"] for o in origins},
                             names=["Forecast start", "Month"]).reindex(results.index)
    assert tuned_values.notna().all()
    display(pd.Series({"Frozen": errors["Target only"].mean(),
                       "Fine-tuned": (tuned_values - results["Observed"]).abs().mean()},
                      name="Target-only MAE (m)").round(2))

**Resources** · [Chronos forecasting examples](https://github.com/amazon-science/chronos-forecasting) · [Chronos-2 model card](https://huggingface.co/amazon/chronos-2) · [Darts foundation-model examples](https://unit8co.github.io/darts/examples/25-FoundationModel-examples.html)